# Research Crew with CrewAI
# Topic - Should I replace my data scientist with an AI agent?

A team of AI agents that work like junior data scientists — they research, analyze, and report on any topic you give them.

**The Crew:**
1. **Data Analyst** — explores data and identifies trends
2. **Insights Strategist** — interprets findings and explains the "so what"
3. **Report Writer** — synthesizes everything into a stakeholder-ready summary

---

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

import os
assert os.environ.get("SERPER_API_KEY")

## Setup

Configure the LLM. By default this uses Ollama (local).

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from crewai_tools import SerperDevTool

llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

print("\u2705 Setup complete!")

## Tools

We give the crew two tools:
- **SerperDevTool** — web search (requires `SERPER_API_KEY` env var)
- **Calculator** — a custom tool for quick math

In [ ]:
search_tool = SerperDevTool()

@tool("Calculator")
def calculator(expression: str) -> str:
    """Perform a mathematical calculation.
    Input should be a math expression like '100 * 0.15' or '(500 - 200) / 3'.
    """
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

print("\u2705 Tools ready!")

---

## Define the Agents

Three specialists, each with a clear role, goal, and background

In [ ]:
data_analyst = Agent(
    role="Data Analyst",
    goal="Research {topic} and identify key data points, trends, and numbers",
    backstory="""You are a detail-oriented data analyst who loves digging into
    numbers. You search for relevant data, spot patterns, and quantify trends.
    You always back up claims with specific figures when possible.""",
    tools=[search_tool, calculator],
    verbose=True,
    llm=llm,
    allow_delegation=False
)

insights_strategist = Agent(
    role="Insights Strategist",
    goal="Interpret the data on {topic} and explain what it means for decision-makers",
    backstory="""You are a strategic thinker who translates raw data into
    actionable insights. You answer the 'so what?' question — why should
    stakeholders care, and what should they do about it?""",
    verbose=True,
    llm=llm,
    allow_delegation=False
)

report_writer = Agent(
    role="Report Writer",
    goal="Create a clear, professional summary report on {topic}",
    backstory="""You are a skilled communicator who writes concise reports
    for busy stakeholders. You structure information logically, lead with
    the bottom line, and use plain language.""",
    verbose=True,
    llm=llm,
    allow_delegation=False
)

print("\u2705 Agents created!")
print(f"   - {data_analyst.role}")
print(f"   - {insights_strategist.role}")
print(f"   - {report_writer.role}")

## Define the Tasks



In [ ]:
analysis_task = Task(
    description="""Research and analyze: {topic}

    Your deliverables:
    - Key statistics and data points (with sources where possible)
    - Notable trends (growth rates, market shifts, adoption curves)
    - Comparison data (competitors, benchmarks, before/after)
    - Any surprising or counterintuitive findings

    Use the search tool to find recent data. Use the calculator
    for any percentage changes or projections.""",
    agent=data_analyst,
    expected_output="A data brief with key statistics, trends, and comparisons"
)

insights_task = Task(
    description="""Review the data analysis and extract strategic insights on {topic}.

    Your deliverables:
    - Top 3 takeaways (the 'so what?' for each data point)
    - Opportunities and risks
    - A recommended action or next step for someone entering this space

    Focus on making the data meaningful, not just repeating numbers.""",
    agent=insights_strategist,
    expected_output="Strategic insights with actionable takeaways",
    context=[analysis_task]
)

report_task = Task(
    description="""Write a stakeholder-ready summary report on {topic}.

    Structure:
    1. **Bottom Line** — one-sentence summary of the most important finding
    2. **Key Data** — 3-5 bullet points with the most compelling numbers
    3. **Strategic Insights** — what the data means and why it matters
    4. **Recommendation** — one clear next step

    Keep it under 400 words. Write for a busy executive.""",
    agent=report_writer,
    expected_output="A concise executive summary report (under 400 words)",
    context=[analysis_task, insights_task]
)

print("\u2705 Tasks defined!")
print("   1. Data Analysis")
print("   2. Strategic Insights (uses analysis)")
print("   3. Executive Report (uses both)")

## Assemble the Crew

In [ ]:
ds_crew = Crew(
    agents=[data_analyst, insights_strategist, report_writer],
    tasks=[analysis_task, insights_task, report_task],
    verbose=True
)

print("\u2705 Data Science Crew assembled!")
print("   Pipeline: Analyze \u2192 Interpret \u2192 Report")

## Run the Crew

In [ ]:
# Change this to any topic you want to analyze
topic = "Should I replace my data scientist with an AI agent?"

print(f"\U0001f4ca Topic: {topic}")
print(f"\U0001f465 Crew: {data_analyst.role} \u2192 {insights_strategist.role} \u2192 {report_writer.role}")
print("\n" + "="*70 + "\n")

result = ds_crew.kickoff(inputs={"topic": topic})

print("\n" + "="*70)
print("FINAL REPORT")
print("="*70 + "\n")
print(result)

## Show Result as a Markdown

In [ ]:
#Final Report
from IPython.display import display, Markdown
display(Markdown(result.raw))